# Step 1: Import necessary libraries

In [10]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, Column, Float, Integer, String, MetaData, Table, text
from sqlalchemy.orm import sessionmaker
from bokeh.plotting import figure, output_file, save, show
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource
import unittest
import os
from sklearn.metrics import mean_squared_error

# Configuration
DB_NAME = "assignment_database.db"
TRAIN_FILE = 'train (1).csv'
IDEAL_FILE = 'ideal (1).csv'
TEST_FILE = 'test (1).csv'

# Step 2: Load data into DataFrames

In [4]:
train_data = pd.read_csv('/content/train (1).csv')
ideal_functions = pd.read_csv('/content/ideal (1).csv')
test_data = pd.read_csv('/content/test (1).csv')

# Check the first few rows of the data
train_data.head(), ideal_functions.head(), test_data.head()


(      x         y1         y2         y3        y4
 0 -20.0  39.778572 -40.078590 -20.214268 -0.324914
 1 -19.9  39.604813 -39.784000 -20.070950 -0.058820
 2 -19.8  40.099070 -40.018845 -19.906782 -0.451830
 3 -19.7  40.151100 -39.518402 -19.389118 -0.612044
 4 -19.6  39.795662 -39.360065 -19.815890 -0.306076,
       x        y1        y2        y3        y4        y5        y6        y7  \
 0 -20.0 -0.912945  0.408082  9.087055  5.408082 -9.087055  0.912945 -0.839071   
 1 -19.9 -0.867644  0.497186  9.132356  5.497186 -9.132356  0.867644 -0.865213   
 2 -19.8 -0.813674  0.581322  9.186326  5.581322 -9.186326  0.813674 -0.889191   
 3 -19.7 -0.751573  0.659649  9.248426  5.659649 -9.248426  0.751573 -0.910947   
 4 -19.6 -0.681964  0.731386  9.318036  5.731386 -9.318036  0.681964 -0.930426   
 
          y8        y9  ...        y41        y42       y43       y44  \
 0 -0.850919  0.816164  ... -40.456474  40.204040  2.995732 -0.008333   
 1  0.168518  0.994372  ... -40.233820  40.0485

# Step 3: Set up SQLite database using sqlalchemy

In [8]:

engine = create_engine('sqlite:///functions.db')

# Store training data and ideal functions in the SQLite database
train_data.to_sql('training_data', engine, if_exists='replace', index=False)
ideal_functions.to_sql('ideal_functions', engine, if_exists='replace', index=False)

# Verify if the tables are stored
with engine.connect() as conn:
    result = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table';"))
    tables = result.fetchall()

tables

[('training_data',), ('ideal_functions',)]

# Step 4: Function to select the 4 best ideal functions based on least-squares error

In [12]:

def select_best_functions(train_data, ideal_functions):
    best_functions = []

    # Iterate through each ideal function column
    for func in ideal_functions.columns[1:]:  # Exclude the 'X' column
        # Calculate least squares error for each function
        # Assuming 'y1' from train_data is the target for comparison
        y_actual = train_data['y1']
        y_pred = ideal_functions[func]
        mse = mean_squared_error(y_actual, y_pred)
        best_functions.append((func, mse))

    # Sort by error (least to greatest)
    best_functions.sort(key=lambda x: x[1])
    return best_functions[:4]  # Return the top 4 functions

# Test the function
best_functions = select_best_functions(train_data, ideal_functions)
best_functions

[('y42', 0.08561648575842126),
 ('y14', 132.99371298082846),
 ('y15', 307.88660385781895),
 ('y1', 531.4721758281776)]

# Step 5: Map training data points to the selected ideal functions

In [15]:

def map_test_data_to_functions(test_data, ideal_functions, best_function_names, max_deviations_train):
    mapped_results = []
    ideal_functions_sorted = ideal_functions.sort_values(by='x').set_index('x')

    # Iterate over the test data
    for i, row in test_data.iterrows():
        x_value = row['x']
        y_value = row['y']

        if x_value not in ideal_functions_sorted.index:
            continue

        best_match_for_point = None
        min_deviation_for_point = float('inf')

        # Check each of the 4 best ideal functions
        for func_name in best_function_names:
            ideal_y = ideal_functions_sorted.loc[x_value, func_name]
            deviation = abs(y_value - ideal_y)

            # Apply deviation threshold (max training deviation * sqrt(2))
            # Make sure to get the max_deviation for the specific ideal function
            threshold = max_deviations_train.get(func_name, float('inf')) * np.sqrt(2)

            if deviation <= threshold and deviation < min_deviation_for_point:
                min_deviation_for_point = deviation
                best_match_for_point = func_name

        if best_match_for_point:
            mapped_results.append({
                'x_test': x_value,
                'y_test': y_value,
                'mapped_to_ideal_function': best_match_for_point,
                'deviation': min_deviation_for_point
            })
    return pd.DataFrame(mapped_results)

# Calculate the max deviation for each best ideal function from the training data
max_deviations_train = {}
for func_name in best_function_names:
    func_mapped_points = mapped_training_points[mapped_training_points['mapped_to_ideal_function'] == func_name]
    if not func_mapped_points.empty:
        max_deviations_train[func_name] = func_mapped_points['deviation'].max()
    else:
        max_deviations_train[func_name] = 0 # Or a suitable default/error handling

# Perform the mapping for test data
mapped_test_points = map_test_data_to_functions(test_data, ideal_functions, best_function_names, max_deviations_train)

# Display the first 5 results and a summary
print("Mapped Test Data Points (first 5):")
print(mapped_test_points.head())
print(f"\nTotal number of test data points: {len(test_data)}")
print(f"Number of successfully mapped test points: {len(mapped_test_points)}")
print(f"Number of unmapped test points: {len(test_data) - len(mapped_test_points)}")

Mapped Test Data Points (first 5):
   x_test     y_test mapped_to_ideal_function  deviation
0   -15.0  -0.205363                       y1   0.444924
1     8.1 -16.659458                      y42   0.337686
2     4.5  -0.840115                       y1   0.137415
3    -8.8  16.571745                      y42   0.622709
4   -17.9   1.169216                       y1   0.356059

Total number of test data points: 100
Number of successfully mapped test points: 19
Number of unmapped test points: 81


In [14]:
def map_train_to_ideal(train_data, ideal_functions, best_function_names, max_deviation_threshold):
    mapped_data = []
    # Ensure ideal_functions is sorted by 'x' to allow efficient lookup
    ideal_functions_sorted = ideal_functions.sort_values(by='x').set_index('x')

    for index, row in train_data.iterrows():
        x_train = row['x']

        if x_train not in ideal_functions_sorted.index:
            # If 'x' value from train_data is not directly in ideal_functions, skip this point
            # For this dataset, 'x' values are expected to align, but it's good practice to handle.
            continue

        ideal_y_values_at_x = {
            name: ideal_functions_sorted.loc[x_train, name]
            for name in best_function_names
        }

        # Iterate through the y columns of the training data (y1, y2, y3, y4)
        for y_col in ['y1', 'y2', 'y3', 'y4']:
            y_train = row[y_col]
            min_deviation = float('inf')
            best_fit_function = None

            # Compare the training y with each of the 4 best ideal functions at x_train
            for ideal_func_name, ideal_y_val in ideal_y_values_at_x.items():
                deviation = abs(y_train - ideal_y_val)
                if deviation < min_deviation:
                    min_deviation = deviation
                    best_fit_function = ideal_func_name

            # Check if the best fit is within the allowed deviation threshold
            if best_fit_function and min_deviation <= max_deviation_threshold:
                mapped_data.append({
                    'x_train': x_train,
                    'y_train': y_train,
                    'mapped_to_ideal_function': best_fit_function,
                    'deviation': min_deviation
                })
            # If a point cannot be mapped within the threshold, it is not included in mapped_data

    return pd.DataFrame(mapped_data)

# Extract the names of the 4 best ideal functions
best_function_names = [func[0] for func in best_functions]


MAX_DEVIATION_THRESHOLD = 0.5

# Perform the mapping of training data points
mapped_training_points = map_train_to_ideal(train_data, ideal_functions, best_function_names, MAX_DEVIATION_THRESHOLD)

# Display the first few mapped points and a summary
print("Mapped Training Data Points (first 5):")
print(mapped_training_points.head())
print(f"\nTotal number of training data points (across all y columns): {len(train_data) * 4}")
print(f"Number of successfully mapped points: {len(mapped_training_points)}")
print(f"Number of unmapped points: {(len(train_data) * 4) - len(mapped_training_points)}")


Mapped Training Data Points (first 5):
   x_train    y_train mapped_to_ideal_function  deviation
0    -20.0  39.778572                      y42   0.425468
1    -19.9  39.604813                      y42   0.443777
2    -19.8  40.099070                      y42   0.208410
3    -19.8  -0.451830                       y1   0.361844
4    -19.7  40.151100                      y42   0.421276

Total number of training data points (across all y columns): 1600
Number of successfully mapped points: 614
Number of unmapped points: 986


# Step 6: Store the results in the database

In [17]:

def store_results(mapped_data_df, engine):
    # Rename columns to match the desired database schema
    results_df = mapped_data_df.rename(columns={
        'x_test': 'X',
        'y_test': 'Y',
        'deviation': 'Deviation',
        'mapped_to_ideal_function': 'Ideal Function'
    })
    results_df.to_sql('test_results', engine, if_exists='replace', index=False)

# Store the mapped results in the database, using mapped_test_points
store_results(mapped_test_points, engine)

# Check if the data has been stored
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM test_results LIMIT 5;"))
    stored_results = result.fetchall()

stored_results

[(-15.0, -0.20536347, 'y1', 0.4449244),
 (8.1, -16.659458, 'y42', 0.3376860000000015),
 (4.5, -0.8401146, 'y1', 0.1374154999999999),
 (-8.8, 16.571745, 'y42', 0.6227090000000004),
 (-17.9, 1.1692159, 'y1', 0.35605876000000003)]

# Step 7: Visualize training data and ideal functions

In [20]:
from bokeh.io import output_notebook

def visualize_training_vs_ideal(train_data, ideal_functions):
    output_notebook() # Configure Bokeh to output to the notebook
    p = figure(title="Training Data and Ideal Functions", x_axis_label='X', y_axis_label='Y')

    # Plot training data (using 'y1' as a representative training series)
    p.line(train_data['x'], train_data['y1'], legend_label="Training Data (y1)", line_width=2, color="blue")

    # Plot each of the ideal functions (just plot the first 4 for example)
    # The ideal_functions DataFrame has 'x' as the first column, followed by 'y1', 'y2', etc.
    # So, ideal_functions.columns[i] for i from 1 to 4 will correctly pick 'y1' through 'y4'.
    for i in range(1, 5):
        p.line(ideal_functions['x'], ideal_functions[ideal_functions.columns[i]], legend_label=f"Ideal Function {ideal_functions.columns[i]}", line_width=2)

    show(p)

# Run the visualization
visualize_training_vs_ideal(train_data, ideal_functions)

# Step 8: Visualize the test data mapped to ideal functions

In [23]:
from bokeh.io import output_notebook

def visualize_test_data_mapped(test_data, best_functions, engine):
    output_notebook() # Ensure plot is displayed inline
    mapped_results = map_test_data_to_functions(test_data, ideal_functions, best_function_names, max_deviations_train)
    results_df = pd.DataFrame(mapped_results, columns=['x_test', 'y_test', 'deviation', 'mapped_to_ideal_function'])

    p = figure(title="Test Data Mapped to Ideal Functions", x_axis_label='X', y_axis_label='Y')

    # Plot the test data and its mapping
    for func_name in best_function_names:
        results = results_df[results_df['mapped_to_ideal_function'] == func_name]
        p.scatter(results['x_test'], results['y_test'], size=8, legend_label=f"Mapped to {func_name}")

    show(p)

# Run the visualization for test data
visualize_test_data_mapped(test_data, best_functions, engine)

# Step 9: Create unit tests

In [25]:

class TestFunctions(unittest.TestCase):
    def test_least_squares(self):
        # Test the function selection process
        best_functions = select_best_functions(train_data, ideal_functions)
        self.assertTrue(len(best_functions) == 4)  # Check that 4 functions are selected

    def test_deviation_mapping(self):
        # Test the mapping process
        best_functions = select_best_functions(train_data, ideal_functions)
        best_function_names_test = [func[0] for func in best_functions] # Extract names for the test

        mapped_results = map_test_data_to_functions(test_data, ideal_functions, best_function_names_test, max_deviations_train)
        self.assertGreater(len(mapped_results), 0)  # Ensure some data is mapped

# Run unit tests
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(TestFunctions))


..
----------------------------------------------------------------------
Ran 2 tests in 0.088s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

#Step 10 Git Commands Task


git init  # Initialize Git in your project folder

git remote add origin https://github.com/your_username/your_repository_name.git  # Add remote URL

git add .  # Stage all files
git commit -m "Initial commit of DLMDSPWP01 assignment project"  # Commit changes

git push -u origin master  # Push to the master branch

git checkout -b develop  # Create and switch to a new branch

git add .  # Stage new/modified files
git commit -m "Added function to map test data to ideal functions"  # Commit changes

git push -u origin develop  # Push the develop branch

git checkout master  # Switch to master branch
git merge develop  # Merge changes from develop to master
git push origin master  # Push merged changes to GitHub

